### Setup

In [5]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# ── Project root detection ──
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DB_PATH = PROJECT_ROOT / "data" / "processed" / "recipeiq.duckdb"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

### Loading Recipe Data

In [6]:
con = duckdb.connect(str(DB_PATH), read_only=True)

recipes = con.execute("""
    SELECT
        RecipeId, Name, RecipeCategory,
        AggregatedRating, ReviewCount,
        Description,
        ingredient_count, complexity_score,
        calories_per_serving
    FROM recipes
    WHERE Description IS NOT NULL
      AND AggregatedRating IS NOT NULL
      AND ReviewCount >= 3
    ORDER BY ReviewCount DESC
    LIMIT 50000
""").fetchdf()

con.close()
print(f"Loaded {len(recipes):,} recipes for recommendation")
print(f"Sample description: {recipes['Description'].iloc[0][:200]}")

Loaded 50,000 recipes for recommendation
Sample description: I searched and finally found this recipe on the internet. It is a copycat of the Bourbon Chicken sold in Chinese carry-outs in my hometown.  This recipe is so good that my sons gobble it up leaving me


### Content-Based — TF-IDF on Descriptions

In [7]:
# Build TF-IDF matrix
tfidf = TfidfVectorizer(
    max_features=5000,      # Top 5000 most common terms
    stop_words="english",   # Remove "the", "a", etc.
    ngram_range=(1, 2),     # Single words + bigrams ("olive oil")
)

tfidf_matrix = tfidf.fit_transform(recipes["Description"].fillna(""))
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Top features: {tfidf.get_feature_names_out()[:20]}")

TF-IDF matrix shape: (50000, 5000)
Top features: ['05' '06' '07' '08' '09' '10' '10 minutes' '10 years' '100' '11' '12'
 '13' '14' '15' '15 minutes' '15 years' '16' '18' '1998' '1st']


### Build the Similarity Function

In [ ]:
def recommend_similar(recipe_name, n=10):
    """
    Find n recipes most similar to the given recipe, based on ingredients.
    """

    # Find the recipe
    matches = recipes[recipes["Name"].str.contains(recipe_name, case=False, na=False)]
    if matches.empty:
        print(f"No recipe found matching '{recipe_name}'")
        return pd.DataFrame()

    idx = matches.index[0]  # Use the first match
    recipe = recipes.loc[idx]
    print(f"Finding recipes similar to: {recipe['Name']} ({recipe['RecipeCategory']})")

    # Compute cosine similarity between this recipe and all others
    similarities = cosine_similarity(tfidf_matrix[idx:idx+1], tfidf_matrix).flatten()

    # Get top-n most similar (excluding the recipe itself)
    similar_indices = similarities.argsort()[::-1][1:n+1]

    results = recipes.iloc[similar_indices][["Name", "RecipeCategory", "AggregatedRating", "ReviewCount"]].copy()
    results["Similarity"] = similarities[similar_indices]

    return results


# Test it
print("="*60)
recommend_similar("Banana Bread", n=10)

Finding recipes similar to: Best Banana Bread (Quick Breads)


,Name,RecipeCategory,AggregatedRating,ReviewCount,Similarity
39197,Best Banana Bread,Breads,5.0,7.0,1.000000
1,Best Banana Bread,Quick Breads,5.0,2273.0,1.000000
39882,Best Banana Bread Or Muffins,Quick Breads,4.5,7.0,0.800085
29337,Peanutty Chocolate Banana Bread,Breads,5.0,9.0,0.787384
37369,No-Butter Banana Bread,Quick Breads,5.0,7.0,0.787373
43472,Mom's Banana Bread,Breads,5.0,7.0,0.776060
35918,Famous Banana Bread,Quick Breads,5.0,8.0,0.743569
3456,Applesauce Banana Bread,Quick Breads,4.5,44.0,0.730110
1343,Banana Bread,Quick Breads,5.0,86.0,0.729761
6498,Banana Bread,Quick Breads,5.0,28.0,0.729761


### Collaborative Filtering

In [9]:
# Load user-recipe rating data
con = duckdb.connect(str(DB_PATH), read_only=True)

reviews = con.execute("""
    SELECT
        AuthorId,
        RecipeId,
        Rating
    FROM reviews
    WHERE Rating IS NOT NULL
      AND RecipeId IN (SELECT RecipeId FROM recipes WHERE ReviewCount >= 3)
""").fetchdf()

con.close()
print(f"Loaded {len(reviews):,} ratings from {reviews['AuthorId'].nunique():,} users")

# Only keep users with at least 5 ratings (need enough history to find patterns)
active_users = reviews["AuthorId"].value_counts()
active_users = active_users[active_users >= 5].index
reviews_filtered = reviews[reviews["AuthorId"].isin(active_users)]
print(f"Active users (5+ ratings): {len(active_users):,}")
print(f"Ratings from active users: {len(reviews_filtered):,}")

Loaded 1,194,187 ratings from 248,443 users
Active users (5+ ratings): 25,147
Ratings from active users: 909,401


### User-Item Matrix + Similarity

In [10]:
from scipy.sparse import csr_matrix

# Create user-item rating matrix
user_item = reviews_filtered.pivot_table(
    index="AuthorId",
    columns="RecipeId",
    values="Rating",
    aggfunc="mean",
).fillna(0)

print(f"User-Item matrix: {user_item.shape}")
print(f"Sparsity: {(user_item == 0).sum().sum() / user_item.size:.2%}")

# Convert to sparse matrix for efficiency
user_item_sparse = csr_matrix(user_item.values)

# Item-item similarity (transpose → rows are recipes)
item_similarity = cosine_similarity(user_item_sparse.T)
print(f"Item similarity matrix: {item_similarity.shape}")

/var/folders/mm/myykphxn7d9415wx0c8yhk2w0000gn/T/ipykernel_62140/3963601447.py:4: PerformanceWarning: The following operation may generate 2841409824 cells in the resulting pandas object.
  user_item = reviews_filtered.pivot_table(


User-Item matrix: (25147, 112992)
Sparsity: 99.97%
Item similarity matrix: (112992, 112992)


### Collaborative Recommender

In [11]:
def recommend_collaborative(recipe_id, n=10):
    """
    Recommend recipes based on user rating patterns.
    "Users who rated this recipe also rated these recipes highly."
    """
    if recipe_id not in user_item.columns:
        print(f"Recipe {recipe_id} not in user-item matrix")
        return pd.DataFrame()

    # Find the column index
    col_idx = user_item.columns.get_loc(recipe_id)

    # Get similarity scores for this recipe
    sim_scores = item_similarity[col_idx]

    # Get top-n most similar (excluding self)
    similar_indices = sim_scores.argsort()[::-1][1:n+1]
    similar_recipe_ids = user_item.columns[similar_indices]
    similar_scores = sim_scores[similar_indices]

    # Look up recipe names
    results = recipes[recipes["RecipeId"].isin(similar_recipe_ids)][
        ["RecipeId", "Name", "RecipeCategory", "AggregatedRating", "ReviewCount"]
    ].copy()

    score_map = dict(zip(similar_recipe_ids, similar_scores))
    results["Similarity"] = results["RecipeId"].map(score_map)
    results = results.sort_values("Similarity", ascending=False)

    return results


# Test: find a popular recipe and get collaborative recommendations
popular_recipe = recipes.iloc[0]
print(f"Collaborative recommendations for: {popular_recipe['Name']}")
recommend_collaborative(popular_recipe["RecipeId"], n=10)

Collaborative recommendations for: Bourbon Chicken


,RecipeId,Name,RecipeCategory,AggregatedRating,ReviewCount,Similarity
19,28148,Oven-Fried Chicken Chimichangas,One Dish Meal,5.0,835.0,0.194092
4,39087,Creamy Cajun Chicken Pasta,Chicken Breast,5.0,1586.0,0.193062
2,27208,To Die for Crock Pot Roast,One Dish Meal,5.0,1692.0,0.186259
35,95222,Pork Chops Yum-Yum,Pork,4.5,681.0,0.174418
3,89204,Crock-Pot Chicken With Black Beans & Cream Cheese,One Dish Meal,4.5,1657.0,0.174187
8,22782,Jo Mama's World Famous Spaghetti,Spaghetti,5.0,1326.0,0.165490
9,32204,"""Whatever Floats Your Boat"" Brownies!",Bar Cookie,5.0,1284.0,0.159513
14,68955,Japanese Mum's Chicken,Chicken Thigh & Leg,5.0,973.0,0.157695
12,69173,Kittencal's Italian Melt-In-Your-Mouth Meatballs,Meat,5.0,1068.0,0.155445
16,82102,Kittencal's Moist Cheddar-Garlic Oven Fried Ch...,Chicken Breast,5.0,908.0,0.154602


### Save the Recommender

In [12]:
import joblib

model_path = MODELS_DIR

# Save TF-IDF model + matrix for content-based
joblib.dump({
    "tfidf": tfidf,
    "tfidf_matrix": tfidf_matrix,
    "recipe_ids": recipes["RecipeId"].values,
    "recipe_names": recipes["Name"].values,
}, model_path / "recommender_tfidf.joblib")

print("Saved content-based recommender")

Saved content-based recommender
